# PRÁCTICA 4 — NOTEBOOK 1: CALIDAD DE DATOS
**Dataset:** Delitos Informáticos en Colombia  
**Objetivo:** Perfilado, diagnóstico y limpieza de datos  
**Variable objetivo:** `CRIMINALIDAD` (SI / NO)

## 0. Instalación e importación de librerías

In [ ]:
# Instalar librerías necesarias (solo primera vez)
!pip install ydata-profiling -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

print('✅ Librerías importadas correctamente')

---
## 1. CARGA DE DATOS ORIGINALES

In [ ]:
# Cargar el dataset original (con tildes, sin limpiar)
df_original = pd.read_csv('Delitos_Informaticos_V1_20260216.csv', low_memory=False)

print(f'📋 Dimensiones del dataset: {df_original.shape[0]:,} filas × {df_original.shape[1]} columnas')
print(f'📝 Columnas: {list(df_original.columns)}')
df_original.head()

---
## 2. PERFILADO DE DATOS (ydata-profiling)

Se genera un reporte automático HTML con estadísticas descriptivas, distribuciones, correlaciones, valores únicos y alertas de calidad para cada columna del dataset original.

In [ ]:
# ⏳ Este proceso puede tardar 2-3 minutos con 63k registros
print('Generando perfil de datos... Por favor espera...')

profile = ProfileReport(
    df_original,
    title='Perfil de Calidad de Datos — Delitos Informáticos Colombia',
    explorative=True,
    minimal=False
)

# Guardar el reporte en HTML (entregable requerido)
profile.to_file('reporte_perfilado_datos.html')
print('✅ Reporte HTML guardado como: reporte_perfilado_datos.html')

In [ ]:
# Mostrar el reporte dentro del notebook (opcional)
profile.to_notebook_iframe()

---
## 3. DIAGNÓSTICO DE CALIDAD DE DATOS

Se evalúan las **5 dimensiones de calidad de datos** basándonos en los resultados del perfilado.

### 3.1 Vista general del dataset

In [ ]:
print('='*60)
print('RESUMEN GENERAL DEL DATASET')
print('='*60)
print(f'Filas totales:     {df_original.shape[0]:,}')
print(f'Columnas totales:  {df_original.shape[1]}')
print(f'Duplicados:        {df_original.duplicated().sum():,}')
print(f'Nulos totales:     {df_original.isnull().sum().sum():,}')
print()
print('Tipos de datos:')
print(df_original.dtypes)

### 3.2 Dimensión 1 — COMPLETITUD
> ¿Existen valores nulos o faltantes en el dataset?

In [ ]:
nulos = df_original.isnull().sum()
pct_nulos = (nulos / len(df_original)) * 100

completitud = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': pct_nulos.round(2),
    'Completitud (%)': (100 - pct_nulos).round(2)
})

print('DIMENSIÓN 1 — COMPLETITUD')
print(completitud.to_string())
print()

completitud_global = 100 - (df_original.isnull().sum().sum() / (df_original.shape[0] * df_original.shape[1]) * 100)
print(f'✅ Completitud global del dataset: {completitud_global:.2f}%')

if completitud_global == 100:
    print('📌 Diagnóstico: El dataset NO presenta valores nulos. Completitud perfecta.')
else:
    print(f'📌 Diagnóstico: Hay datos faltantes en algunas columnas. Se requiere imputación.')

In [ ]:
# Gráfico de completitud
fig, ax = plt.subplots(figsize=(10, 5))
completitud['Completitud (%)'].sort_values().plot(
    kind='barh', ax=ax, color='steelblue', edgecolor='white'
)
ax.axvline(x=100, color='green', linestyle='--', label='100% completo')
ax.set_title('Completitud por columna (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Completitud (%)')
ax.set_xlim(0, 105)
for i, v in enumerate(completitud['Completitud (%)'].sort_values()):
    ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('completitud.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como completitud.png')

### 3.3 Dimensión 2 — UNICIDAD
> ¿Existen registros duplicados en el dataset?

In [ ]:
duplicados = df_original.duplicated().sum()
pct_dup = (duplicados / len(df_original)) * 100

print('DIMENSIÓN 2 — UNICIDAD')
print(f'Registros totales:     {len(df_original):,}')
print(f'Registros duplicados:  {duplicados:,} ({pct_dup:.2f}%)')
print(f'Registros únicos:      {len(df_original) - duplicados:,}')
print()

# Columnas con baja cardinalidad (muchos valores repetidos)
print('Valores únicos por columna (cardinalidad):')
for col in df_original.columns:
    nunique = df_original[col].nunique()
    pct = nunique / len(df_original) * 100
    flag = '⚠️ MUY BAJA CARDINALIDAD' if nunique == 1 else ('📌 Baja' if pct < 0.1 else '')
    print(f'  {col:<25} → {nunique:>5} únicos ({pct:.3f}%) {flag}')

print()
if duplicados == 0:
    print('✅ Diagnóstico: No se detectaron filas duplicadas.')
else:
    print(f'⚠️ Diagnóstico: Se encontraron {duplicados:,} registros duplicados. Deben eliminarse.')

### 3.4 Dimensión 3 — CONSISTENCIA
> ¿Los datos son coherentes internamente? ¿Hay inconsistencias lógicas entre columnas?

In [ ]:
print('DIMENSIÓN 3 — CONSISTENCIA')
print()

# Verificación 1: Columnas con un solo valor único (sin variabilidad)
print('--- Columnas con un solo valor único (sin variabilidad informativa) ---')
for col in df_original.columns:
    if df_original[col].nunique() == 1:
        val = df_original[col].unique()[0]
        print(f'  ⚠️  {col}: solo contiene "{val}" en {len(df_original):,} registros → columna constante')

# Verificación 2: Año de hechos vs año de entrada
print()
print('--- Consistencia temporal: AÑO_HECHOS vs AÑO_ENTRADA ---')
df_original['AÑO_HECHOS_num'] = pd.to_numeric(df_original['AÑO_HECHOS'], errors='coerce')
df_original['AÑO_ENTRADA_num'] = pd.to_numeric(df_original['AÑO_ENTRADA'], errors='coerce')
inconsistentes = df_original[df_original['AÑO_HECHOS_num'] > df_original['AÑO_ENTRADA_num']]
print(f'  Registros donde AÑO_HECHOS > AÑO_ENTRADA: {len(inconsistentes):,}')
if len(inconsistentes) > 0:
    print('  ⚠️  Inconsistencia lógica: el hecho ocurre DESPUÉS del ingreso al sistema')
else:
    print('  ✅ Consistencia temporal correcta')

# Limpieza de columnas auxiliares
df_original.drop(columns=['AÑO_HECHOS_num','AÑO_ENTRADA_num'], inplace=True)

# Verificación 3: Tildes y codificación en nombres de columnas
print()
print('--- Caracteres especiales en nombres de columnas ---')
cols_con_tildes = [c for c in df_original.columns if any(ch in c for ch in 'ÁÉÍÓÚÑÜ')]
if cols_con_tildes:
    print(f'  ⚠️  Columnas con tildes/caracteres especiales: {cols_con_tildes}')
    print('  → Estos nombres pueden causar errores en sistemas externos y deben normalizarse.')
else:
    print('  ✅ No se encontraron caracteres especiales en nombres de columnas.')

### 3.5 Dimensión 4 — VALIDEZ
> ¿Los valores están dentro de rangos y formatos esperados?

In [ ]:
print('DIMENSIÓN 4 — VALIDEZ')
print()

# Validar años (rango esperado: 2000 - año actual)
print('--- Rango de años ---')
for col_anio in ['AÑO_HECHOS', 'AÑO_ENTRADA']:
    serie = pd.to_numeric(df_original[col_anio], errors='coerce')
    fuera_rango = ((serie < 2000) | (serie > 2026)).sum()
    print(f'  {col_anio}: min={serie.min()}, max={serie.max()}, '
          f'fuera de rango [2000-2026]: {fuera_rango}')
    if fuera_rango == 0:
        print(f'    ✅ Valores válidos')
    else:
        print(f'    ⚠️ {fuera_rango} registros fuera del rango esperado')

# Validar columnas binarias (SI/NO)
print()
print('--- Columnas binarias (valores esperados: SI / NO) ---')
binarias = ['CRIMINALIDAD', 'ES_ARCHIVO', 'ES_PRECLUSIÓN']
for col in binarias:
    vals = df_original[col].unique()
    invalidos = [v for v in vals if v not in ['SI', 'NO']]
    print(f'  {col}: valores encontrados → {list(vals)}')
    if invalidos:
        print(f'    ⚠️ Valores inválidos: {invalidos}')
    else:
        print(f'    ✅ Valores válidos')

# Validar TOTAL_PROCESOS (debe ser > 0)
print()
print('--- TOTAL_PROCESOS (debe ser > 0) ---')
negativos = (df_original['TOTAL_PROCESOS'] <= 0).sum()
print(f'  Registros con TOTAL_PROCESOS <= 0: {negativos}')
if negativos == 0:
    print('  ✅ Todos los valores son positivos')
else:
    print(f'  ⚠️ {negativos} registros con valores inválidos')

### 3.6 Dimensión 5 — EXACTITUD
> ¿Los datos representan correctamente la realidad? ¿Existen valores atípicos (outliers)?

In [ ]:
print('DIMENSIÓN 5 — EXACTITUD')
print()

# Análisis de outliers en TOTAL_PROCESOS (única columna numérica relevante)
col = 'TOTAL_PROCESOS'
serie = df_original[col]
Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR
outliers = ((serie < lim_inf) | (serie > lim_sup)).sum()

print(f'Columna: {col}')
print(f'  Media:         {serie.mean():.2f}')
print(f'  Mediana:       {serie.median():.2f}')
print(f'  Desv. típica:  {serie.std():.2f}')
print(f'  Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}')
print(f'  Límite inf.:   {lim_inf:.2f}')
print(f'  Límite sup.:   {lim_sup:.2f}')
print(f'  Outliers (IQR): {outliers:,} ({outliers/len(serie)*100:.2f}%)')
print(f'  Valor máximo:  {serie.max():,}')

# Verificar valores extraños en PAIS_HECHO
print()
print(f'País hecho - valores únicos: {df_original["PAÍS_HECHO"].unique()}')
sin_dato_pais = (df_original['PAÍS_HECHO'] == 'SIN DATO').sum()
print(f'  Registros con "SIN DATO" en PAÍS_HECHO: {sin_dato_pais:,} ({sin_dato_pais/len(df_original)*100:.2f}%)')

In [ ]:
# Visualización de outliers
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot
axes[0].boxplot(df_original['TOTAL_PROCESOS'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[0].set_title('Distribución TOTAL_PROCESOS\n(con outliers)', fontweight='bold')
axes[0].set_ylabel('Total de Procesos')

# Histograma
axes[1].hist(df_original['TOTAL_PROCESOS'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(lim_sup, color='red', linestyle='--', label=f'Límite sup. IQR = {lim_sup:.0f}')
axes[1].set_title('Histograma TOTAL_PROCESOS', fontweight='bold')
axes[1].set_xlabel('Total de Procesos')
axes[1].legend()

plt.tight_layout()
plt.savefig('outliers_total_procesos.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como outliers_total_procesos.png')

### 3.7 Resumen del Diagnóstico de Calidad

In [ ]:
resumen = pd.DataFrame([
    ['1. Completitud',  '✅ 100%',           'El dataset no presenta valores nulos en ninguna columna.'],
    ['2. Unicidad',     '✅ Sin duplicados',  'No se detectaron registros duplicados exactos.'],
    ['3. Consistencia', '⚠️ Problemas leves', 'GRUPO_DELITO y CONSUMADO tienen un solo valor (constantes). '
                                              'Nombres de columnas tienen tildes (AÑO, PAÍS).'],
    ['4. Validez',      '✅ Válido',          'Los rangos de años y valores binarios (SI/NO) son correctos.'],
    ['5. Exactitud',    '⚠️ Outliers',        'TOTAL_PROCESOS presenta outliers extremos (valores > límite IQR). '
                                              '"SIN DATO" en PAÍS_HECHO en algunos registros.'],
], columns=['Dimensión', 'Estado', 'Descripción'])

print('=== DIAGNÓSTICO DE CALIDAD DE DATOS ===')
print(resumen.to_string(index=False))

---
## 4. LIMPIEZA Y MEJORA DE DATOS

Se implementan todos los pasos de limpieza identificados en el diagnóstico.

### 4.1 Paso 1 — Eliminar columnas constantes (sin información)

In [ ]:
df_limpio = df_original.copy()

# Eliminar columnas con un solo valor único (no aportan al modelo)
cols_constantes = [col for col in df_limpio.columns if df_limpio[col].nunique() == 1]
print(f'Columnas constantes a eliminar: {cols_constantes}')
df_limpio.drop(columns=cols_constantes, inplace=True)
print(f'✅ Eliminadas {len(cols_constantes)} columna(s) constante(s)')
print(f'   Dataset: {df_limpio.shape[0]:,} filas × {df_limpio.shape[1]} columnas')

### 4.2 Paso 2 — Normalizar nombres de columnas (quitar tildes)

In [ ]:
import unicodedata

def normalizar_nombre(nombre):
    """Elimina tildes y caracteres especiales de nombres de columnas."""
    nfkd = unicodedata.normalize('NFKD', nombre)
    sin_tildes = ''.join(c for c in nfkd if not unicodedata.combining(c))
    return sin_tildes.replace(' ', '_').upper()

cols_antes = list(df_limpio.columns)
df_limpio.columns = [normalizar_nombre(c) for c in df_limpio.columns]
cols_despues = list(df_limpio.columns)

print('Renombrado de columnas:')
for a, d in zip(cols_antes, cols_despues):
    cambio = ' → ' + d if a != d else ' ✅ (sin cambio)'
    print(f'  {a}{cambio}')

### 4.3 Paso 3 — Estandarizar tipos de datos

In [ ]:
# Convertir años a entero
for col in ['ANO_HECHOS', 'ANO_ENTRADA']:
    if col in df_limpio.columns:
        df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce').astype('Int64')
        print(f'✅ {col} convertido a Int64')

# Convertir ANO_DENUNCIA a entero
if 'ANO_DENUNCIA' in df_limpio.columns:
    df_limpio['ANO_DENUNCIA'] = pd.to_numeric(df_limpio['ANO_DENUNCIA'], errors='coerce').astype('Int64')
    print('✅ ANO_DENUNCIA convertido a Int64')

# Estandarizar columnas de texto a mayúsculas
cols_texto = df_limpio.select_dtypes(include='object').columns
for col in cols_texto:
    df_limpio[col] = df_limpio[col].str.strip().str.upper()
print(f'✅ {len(cols_texto)} columnas de texto estandarizadas a mayúsculas')

### 4.4 Paso 4 — Tratar outliers en TOTAL_PROCESOS

In [ ]:
# Aplicar capping al percentil 99 (winsorización)
p99 = df_limpio['TOTAL_PROCESOS'].quantile(0.99)
n_outliers = (df_limpio['TOTAL_PROCESOS'] > p99).sum()

print(f'Límite de capping (percentil 99): {p99:.0f}')
print(f'Registros con valores superiores: {n_outliers:,}')

df_limpio['TOTAL_PROCESOS'] = df_limpio['TOTAL_PROCESOS'].clip(upper=p99)

print(f'✅ Outliers tratados con capping (winsorización al P99)')
print(f'   Nuevo máximo: {df_limpio["TOTAL_PROCESOS"].max()}')

### 4.5 Paso 5 — Verificar duplicados y limpiarlos

In [ ]:
n_antes = len(df_limpio)
df_limpio.drop_duplicates(inplace=True)
n_despues = len(df_limpio)

print(f'Registros antes de eliminar duplicados: {n_antes:,}')
print(f'Registros después:                      {n_despues:,}')
print(f'Registros eliminados:                   {n_antes - n_despues:,}')
print(f'✅ Paso completado')

### 4.6 Paso 6 — Crear variable DECADA a partir de AÑO_HECHOS (feature engineering)

In [ ]:
# Crear una variable categórica por período de tiempo
def clasificar_periodo(ano):
    if pd.isna(ano):
        return 'DESCONOCIDO'
    elif ano <= 2015:
        return 'ANTES_2015'
    elif ano <= 2019:
        return '2016_2019'
    elif ano <= 2022:
        return '2020_2022'
    else:
        return '2023_ADELANTE'

df_limpio['PERIODO_HECHO'] = df_limpio['ANO_HECHOS'].apply(clasificar_periodo)
print('✅ Variable PERIODO_HECHO creada:')
print(df_limpio['PERIODO_HECHO'].value_counts())

### 4.7 Resumen final del dataset limpio

In [ ]:
print('='*60)
print('RESUMEN — DATASET LIMPIO vs ORIGINAL')
print('='*60)
print(f'{'Métrica':<30} {'ORIGINAL':>15} {'LIMPIO':>15}')
print('-'*60)
print(f'{'Filas':<30} {df_original.shape[0]:>15,} {df_limpio.shape[0]:>15,}')
print(f'{'Columnas':<30} {df_original.shape[1]:>15} {df_limpio.shape[1]:>15}')
print(f'{'Valores nulos':<30} {df_original.isnull().sum().sum():>15,} {df_limpio.isnull().sum().sum():>15,}')
print(f'{'Duplicados':<30} {df_original.duplicated().sum():>15,} {df_limpio.duplicated().sum():>15,}')
print(f'{'Columnas constantes':<30} {len(cols_constantes):>15} {"0":>15}')
print('='*60)
print()
print('Columnas del dataset limpio:')
print(list(df_limpio.columns))

In [ ]:
# Gráfico de distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución de CRIMINALIDAD
vc = df_limpio['CRIMINALIDAD'].value_counts()
colors = ['#2196F3', '#FF5722']
axes[0].pie(vc, labels=vc.index, autopct='%1.1f%%', colors=colors,
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Distribución Variable Objetivo\nCRIMINALIDAD', fontweight='bold')

# Distribución por departamento (top 10)
top10_dpto = df_limpio['DEPARTAMENTO_HECHO'].value_counts().head(10)
top10_dpto.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Top 10 Departamentos con más\nDelitos Informáticos', fontweight='bold')
axes[1].set_xlabel('Cantidad de registros')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('distribucion_variables.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como distribucion_variables.png')

### 4.8 Guardar dataset limpio

In [ ]:
# Guardar el dataset limpio para usarlo en el Notebook 2
df_limpio.to_csv('Delitos_Informaticos_Limpio.csv', index=False, encoding='utf-8')

print('✅ Dataset limpio guardado como: Delitos_Informaticos_Limpio.csv')
print(f'   Dimensiones finales: {df_limpio.shape[0]:,} filas × {df_limpio.shape[1]} columnas')
print()
print('Primeras filas del dataset limpio:')
df_limpio.head()

---
## ✅ CONCLUSIONES — CALIDAD DE DATOS

| Dimensión | Hallazgo | Acción tomada |
|-----------|----------|---------------|
| **Completitud** | 100% — sin valores nulos | Ninguna |
| **Unicidad** | Sin registros duplicados | Verificado y confirmado |
| **Consistencia** | Columnas constantes (`GRUPO_DELITO`, `CONSUMADO`). Tildes en nombres de columnas | Columnas eliminadas. Nombres normalizados |
| **Validez** | Años y valores binarios dentro de rangos esperados | Tipos de datos estandarizados |
| **Exactitud** | Outliers extremos en `TOTAL_PROCESOS`. `SIN DATO` en `PAIS_HECHO` | Winsorización al percentil 99 aplicada |

El dataset limpio resultante es apto para entrenar modelos de machine learning en el Notebook 2.